In [ ]:
import numpy as np                   #Importing required libraries and datasets
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist


(imgtrain, _), (imgtest, _) = mnist.load_data() #Getting testing and training iamges from mnist dataset


imgtrain = imgtrain.astype("float32") / 255.0 #Normalising them to floating number betweem 0 and 1
imgtest  = imgtest.astype("float32") / 255.0  #Its done so that CNN can be trained easily


imgtrain = np.expand_dims(imgtrain, -1) #Adding channel dimension as expected by CNN
imgtest  = np.expand_dims(imgtest, -1)


preimgtrain = np.load("/content/mnistforegrounddataset/trainmasks.npy") #Laoding previoulsy saved foreground masks
preimgtest  = np.load("/content/mnistforegrounddataset/testmasks.npy")


preimgtrain  = np.expand_dims((preimgtrain  > 0).astype("float32"), -1) #ensuring that images are bianry and converting them to float value
preimgtest  = np.expand_dims((preimgtest > 0).astype("float32"), -1)


print("imgtrain shape:",imgtrain .shape)      #for confirming input and output sizes
print("preimgtrain  shape:", preimgtrain .shape)
print("imgtest shape:",imgtest .shape)
print("preimgtest shape:", preimgtest.shape)


def buildunet(inputshape=(28, 28, 1)): #defining a function to build a model
    inputs = layers.Input(shape=inputshape) #Defining input layer of our Neural network
    #Encoding
    convo1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs) #First convolutional layer
    pool1 = layers.MaxPooling2D()(convo1) #Pooling is done to reduce spatial size so that only imp features remains


    convo2 = layers.Conv2D(64, 3, activation='relu', padding='same')(pool1) #another convolutional layer
    pool2 = layers.MaxPooling2D()(convo2) #another pooling layer


    bot1 = layers.Conv2D(128, 3, activation='relu', padding='same')(pool2) #For getting high level features
    #decoding
    upsamp1 = layers.UpSampling2D()(bot1) #upsampling feature map from bottleneck
    concat1 = layers.Concatenate()([upsamp1, convo2]) #merging upsampled features with encoded features
    convo3 = layers.Conv2D(64, 3, activation='relu', padding='same')(concat1) #Refining merged feature map with convolutional layer


    upsamp2 = layers.UpSampling2D()(convo3) #Upsampling output of last step
    concat2 = layers.Concatenate()([upsamp2, convo1]) #merging done
    convo4 = layers.Conv2D(32, 3, activation='relu', padding='same')(concat2) #Refining done using convolutional layer


    outputs = layers.Conv2D(1, 1, activation='sigmoid')(convo4) #single channel output


    model = models.Model(inputs, outputs) #Wrapping all the layers in model object
    return model


model = buildunet() #printing layer by layer summary
model.summary()


model.compile(optimizer='adam',                #Defining metric , optimizer and loss
              loss='binary_crossentropy',
              metrics=['accuracy'])


history = model.fit(
   imgtrain , preimgtrain ,     #Setting cycles as 5 and batchsize as 100 so that overall accuracy is more and code is not too slow
    validation_data=(imgtest, preimgtest),
    epochs=5,
    batch_size=100
)


def IOUmetric(trueout, predout, threshold=0.5): #Function for getting IOU metrice
    predbinout = (predout> threshold).astype(np.float32) #Converting to binary
    intersection = np.sum(trueout * predbinout) #Total number of correct foreground pixels
    union = np.sum(trueout) + np.sum(predbinout) - intersection #getting union area of both predicted and true outputs
    return intersection / union if union != 0 else 0 #Getting IOU which is intersetion \ union


pred = model.predict(imgtest) #Getting models predicted masks on the input data
iouscore = IOUmetric(preimgtest, pred)
print(f"IoU Score: {iouscore:.4f}")


n = 5 #Plotting sample results
plt.figure(figsize=(10, 4))
for i in range(n):       #Input images
    plt.subplot(3, n, i + 1)
    plt.imshow(imgtest[i].squeeze(), cmap='gray')
    plt.title("Input Image")
    plt.axis('off')


    plt.subplot(3, n, i + 1 + n) #Foreground masks which were already saved
    plt.imshow(preimgtest[i].squeeze(), cmap='gray')
    plt.title("Ground Truth")
    plt.axis('off')


    plt.subplot(3, n, i + 1 + 2*n) #Predicted mask by training model
    plt.imshow(pred[i].squeeze(), cmap='gray')
    plt.title("Predicted Mask")
    plt.axis('off')


plt.tight_layout() #Displaying output
plt.show()